# 01 · Data: Kerala before/after pairs from Google Earth Engine

Builds the training data for `kerala_hazard_vision_ai`:

1. Show each event's before/after Sentinel-2 composite on a map, to check its bounding box.
2. Measure how much of each event area has a cloud-free composite (go / no-go per event).
3. Export an 11-band stack per event to Google Drive (before, after, slope, rain, soil).
4. Cut the stacks into 256×256 tiles with a severity label each.

All logic lives in the `kerala_hazard_vision_ai` package; this notebook only calls it.

**How to run:** Runtime → Run all. It pauses twice for you to approve access (Google Drive,
then Earth Engine): pick the same Google account both times. The whole notebook takes about
30–60 minutes, mostly waiting for Earth Engine exports. Re-running is safe: finished exports
are skipped.
**Before running:** set `GEE_PROJECT` to your Earth Engine Cloud project ID.

In [ ]:
from google.colab import drive

GEE_PROJECT = "practical-range-495917-q8"  # your Earth Engine Cloud project
REPO_URL = "https://github.com/muhammedshahidsds25-collab/kerala-hazard-vision-ai.git"
REPO_DIR = "/content/kerala-hazard-vision-ai"

drive.mount("/content/drive")
!git clone -q {REPO_URL} {REPO_DIR} || git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
# Regular (not editable) install: an editable src-layout install is not importable
# until the runtime restarts.
!pip install -q ".[gee]"

In [ ]:
import logging
from pathlib import Path

import ee
import geemap
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from kerala_hazard_vision_ai.config import load_config
from kerala_hazard_vision_ai.data import gee_export as gx
from kerala_hazard_vision_ai.data.tiling import write_event_tiles

logging.basicConfig(level=logging.INFO)
ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)

cfg = load_config("configs/default.yaml")
DRIVE_ROOT = Path("/content/drive/MyDrive") / cfg.data.drive_folder
TILE_DIR = DRIVE_ROOT / "tiles"
RGB_VIS = {"bands": ["B4", "B3", "B2"], "min": 0.0, "max": 0.3}
[e.name for e in cfg.events]

## 1. Visual check of every event
Toggle the layers: the landslide scar or flood should be inside the red box and visible in
the *after* layer. If a box is off, fix it in `configs/events.yaml`, commit, and re-run.

In [ ]:
# OpenStreetMap blocks tiles requested from Colab, so use Esri satellite imagery instead.
m = geemap.Map(center=(10.5, 76.5), zoom=8, basemap="Esri.WorldImagery")
for event in cfg.events:
    region = gx.aoi(event.bbox)
    (pre_start, pre_end), (post_start, post_end) = gx.composite_windows(event.date, cfg.data)
    m.addLayer(gx.s2_composite(region, pre_start, pre_end, cfg.data.max_cloud_pct), RGB_VIS,
               f"{event.name} before", False)
    m.addLayer(gx.s2_composite(region, post_start, post_end, cfg.data.max_cloud_pct), RGB_VIS,
               f"{event.name} after", False)
    m.addLayer(ee.FeatureCollection([ee.Feature(region)]).style(color="red", fillColor="00000000"),
               {}, f"{event.name} box")
m

## 2. Cloud-free check (go / no-go)
Monsoon clouds are the main data risk. An event is kept only if both composites cover at
least `min_valid_frac` of its area.

**Expected:** `kerala_floods_2018` will probably fail. Its before composite (Dec 2017 - Apr
2018) predates global Sentinel-2 Level-2A coverage in Earth Engine, and flood water has
usually drained by the next dry season. If it fails, floods become future work in the report.

In [ ]:
check = pd.DataFrame(
    [
        {"event": e.name, "hazard": e.hazard, **gx.cloud_free_fraction(e, cfg.data)}
        for e in cfg.events
    ]
)
check["keep"] = (check[["pre", "post"]] >= cfg.data.min_valid_frac).all(axis=1)
check

In [ ]:
kept = [e for e in cfg.events if check.set_index("event").loc[e.name, "keep"]]
print("Kept:", [e.name for e in kept])
print("Dropped:", [e.name for e in cfg.events if e not in kept])
if not any(e.name == cfg.data.test_event for e in kept):
    raise RuntimeError(f"The test event {cfg.data.test_event} failed the cloud check: pick another")

**Flood go / no-go:** _record the result here after running (e.g. "kerala_floods_2018 kept,
flood water visible in the after composite" or "dropped: water receded before the dry-season
composite; floods become future work")._

## 3. Export stacks to Google Drive

In [ ]:
import time


def tif_path(event):
    return DRIVE_ROOT / f"{event.name}.tif"


# Skip events already exported by an earlier run (Drive would otherwise keep duplicates).
to_export = [e for e in kept if not tif_path(e).exists()]
tasks = {e.name: gx.export_event(e, cfg.data) for e in to_export}
print(f"Exporting {len(tasks)} event(s); this usually takes 10-40 minutes...")
while any(t.active() for t in tasks.values()):
    time.sleep(30)
for name, task in tasks.items():
    status = task.status()
    print(name, status["state"], status.get("error_message", ""))

# Finished exports can take a few minutes to appear in the mounted Drive folder.
for _ in range(20):
    if all(tif_path(e).exists() for e in kept):
        break
    time.sleep(15)
ready = [e for e in kept if tif_path(e).exists()]
print("Ready to tile:", [e.name for e in ready])
print("Missing:", [e.name for e in kept if e not in ready])

## 4. Tile the stacks

In [ ]:
counts = {}
for event in ready:
    stack = gx.read_stack(DRIVE_ROOT / f"{event.name}.tif")
    counts[event.name] = write_event_tiles(stack, event, TILE_DIR, cfg)
counts

In [ ]:
severities = pd.DataFrame(
    [
        {"event": str(z["event"]), "severity": float(z["severity"])}
        for z in map(np.load, sorted(TILE_DIR.glob("*.npz")))
    ]
)
print(severities.groupby("event")["severity"].describe())
severities.hist(column="severity", by="event", bins=20, figsize=(12, 6), sharex=True)
plt.tight_layout()

Set `demo.train_severity_max` in `configs/default.yaml` to the value below and commit it:
the demo labels anything higher as extrapolation.

In [ ]:
round(severities["severity"].max(), 3)